In [1]:
import os
from pathlib import Path
from collections import defaultdict, deque

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import IterableDataset, DataLoader
from tqdm.auto import tqdm
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score,
    roc_auc_score, classification_report,
    precision_recall_curve, auc
)


C:\Users\taren\AppData\Roaming\Python\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Test

In [ ]:
# ---------------- user-config -------------------------------------------------
PARQUET_PATH  = Path("MasterDS/Master_cleaned.parquet")
MODEL_WEIGHTS = Path("models/TopSelector_final.pt")
RETURN_COL    = "ret_5d_future"
BATCH_SIZE    = 512          # streamed; only a few samples live in RAM
WINDOW        = 10
DEVICE        = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# -----------------------------------------------------------------------------


# 1) ---------- metadata -------------------------------------------------------
pf = pq.ParquetFile(PARQUET_PATH)
ALL_COLS     = pf.schema_arrow.names
FEATURE_COLS = [c for c in ALL_COLS if c not in {RETURN_COL, "date", "ticker"}]

# reproduce 80 / 20 date split used in training
all_dates = sorted({
    d
    for rg in range(pf.num_row_groups)
    for d  in pf.read_row_group(rg, columns=["date"]).column(0).to_pylist()
})
split = int(len(all_dates) * 0.8)
TEST_DATES = set(all_dates[split:])

# quick StandardScaler (fit on a 10 k-row sample – fine for evaluation)
sample_frames = []
for rg in range(min(10, pf.num_row_groups)):
    df = pf.read_row_group(rg).to_pandas()
    if df.empty:
        continue
    sample_frames.append(df.sample(1000, random_state=0)[FEATURE_COLS])
SCALER = StandardScaler().fit(pd.concat(sample_frames).values)

# per-day 95-th percentile cutoff (needed for labels)
cutoff = defaultdict(list)
for batch in pf.iter_batches(batch_size=2_000_000):
    df = batch.to_pandas()[["date", RETURN_COL]]
    for d, g in df.groupby("date"):
        cutoff[d].extend(g[RETURN_COL].values)
CUTOFF_MAP = {d: np.quantile(v, 0.95) for d, v in cutoff.items()}


# 2) ---------- model (must match training names) ------------------------------
class FeatureTokenizer(nn.Module):
    def __init__(self, n_feat, d=128):
        super().__init__()
        self.proj       = nn.Linear(n_feat, d)
        self.pos_embed  = nn.Parameter(torch.zeros(WINDOW, d))  # <-- SAME NAME
    def forward(self, x):                                       # [B,W,F]
        return self.proj(x) + self.pos_embed


class FTTransformer(nn.Module):
    def __init__(self, n_feat):
        super().__init__()
        d = 128
        self.tok = FeatureTokenizer(n_feat, d)
        self.enc = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(
                d_model=d, nhead=8, dim_feedforward=d*2,
                dropout=0.2, batch_first=True,
                activation="gelu", norm_first=True,
            ),
            num_layers=6,
        )
        self.head = nn.Sequential(nn.LayerNorm(d), nn.Linear(d, 1))
    def forward(self, x):                                       # [B,W,F]
        h = self.enc(self.tok(x))
        return self.head(h.mean(dim=1)).squeeze(1)              # [B]


# 3) ---------- streaming test dataset ----------------------------------------
class ParquetSequenceDataset(IterableDataset):
    def __init__(self, date_subset):
        super().__init__()
        self.dates = set(date_subset)
    def __iter__(self):
        history = defaultdict(lambda: deque(maxlen=WINDOW))
        pf_local = pq.ParquetFile(PARQUET_PATH)
        for rg in range(pf_local.num_row_groups):
            df = pf_local.read_row_group(rg).to_pandas()
            df = df[df["date"].isin(self.dates)]
            if df.empty:
                continue
            df.sort_values(["ticker", "date"], inplace=True)
            df["label"] = (df[RETURN_COL] >= df["date"].map(CUTOFF_MAP)).astype(int)

            for row in df.itertuples(index=False):
                hist = history[row.ticker]
                feats = np.array([getattr(row, c) for c in FEATURE_COLS], np.float32)
                hist.append(feats)
                if len(hist) < WINDOW:
                    continue

                X_seq = np.stack(hist)                                   # [W,F]
                X_seq = SCALER.transform(X_seq).clip(-5, 5)
                yield (
                    torch.from_numpy(X_seq).unsqueeze(0),                # [1,W,F]
                    torch.tensor([float(row.label)]),                    # [1]
                )


# 4) ---------- evaluation loop -----------------------------------------------
def evaluate():
    print(f"Loading model on {DEVICE} …")
    model = FTTransformer(len(FEATURE_COLS)).to(DEVICE)
    model.load_state_dict(torch.load(MODEL_WEIGHTS, map_location=DEVICE))
    model.eval()

    ds      = ParquetSequenceDataset(TEST_DATES)
    loader  = DataLoader(ds, batch_size=None, num_workers=0)

    y_true, y_prob = [], []

    for xb, yb in tqdm(loader, desc="Evaluating", unit="batch", ncols=80):
        xb = xb.to(DEVICE)                 # [1,W,F]
        with torch.no_grad():
            prob = torch.sigmoid(model(xb)).cpu().item()
        y_true.append(int(yb.item()))
        y_prob.append(prob)

    y_true = np.array(y_true, dtype=int)
    y_prob = np.array(y_prob)
    y_pred = (y_prob >= 0.5).astype(int)
    

    print("\n──────────  METRICS  ──────────")
    print("Accuracy       :", accuracy_score(y_true, y_pred))
    print("Balanced Acc   :", balanced_accuracy_score(y_true, y_pred))
    print("AUROC          :", roc_auc_score(y_true, y_prob))
    print(classification_report(y_true, y_pred, digits=3))


# 5) ---------- run ------------------------------------------------------------
if __name__ == "__main__":
    evaluate()

C:\Users\taren\AppData\Roaming\Python\Python312\site-packages\torch\nn\modules\transformer.py:382: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


Loading model on cuda …


Evaluating: 1144736batch [1:16:43, 332.00batch/s]

## Scores

21% Accuracy
